# MicroMind + RAG Geode di Colab (tanpa build)
1. Runtime -> Change runtime type -> **T4 GPU**
2. Colab Secrets: tambah `GITHUB_TOKEN` (scope `repo`) dan aktifkan akses notebook
3. Ubah `API_KEY` di sel konfigurasi, lalu **Run all**

Pakai wheel `llama-cpp-python` CUDA yang sudah jadi (Python 3.12 lewat `uv`, karena wheel CUDA belum ada untuk Python 3.13 bawaan Colab). Library CUDA dipasang lewat pip (`nvidia-cuda-runtime-cu12`, `nvidia-cublas-cu12`).

Catatan: Colab gratis bukan hosting. Sesi bisa putus sewaktu-waktu dan URL tunnel berubah tiap sesi. Aturan Colab membatasi pemakaian sebagai server layanan publik, jadi pakai untuk uji pribadi.

In [ ]:
!pip install -q uv fastapi uvicorn httpx
import os, subprocess, requests, time, re, shutil

V = "/content/venv312"

def sh(c):
    r = subprocess.run(c, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-1200:])
    if r.returncode:
        raise RuntimeError("gagal: " + c[:90])

if not os.path.exists(V + "/bin/python"):
    sh("uv venv --python 3.12 " + V)
    sh("uv pip install --python " + V + "/bin/python --index-strategy unsafe-best-match "
       "--only-binary llama-cpp-python "
       "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 "
       "'llama-cpp-python[server]' nvidia-cuda-runtime-cu12 nvidia-cublas-cu12")

LD = ":".join("%s/lib/python3.12/site-packages/nvidia/%s/lib" % (V, d) for d in ("cuda_runtime", "cublas"))
env = dict(os.environ, LD_LIBRARY_PATH=LD + ":" + os.environ.get("LD_LIBRARY_PATH", ""))
r = subprocess.run([V + "/bin/python", "-c",
                    "import llama_cpp; print('llama-cpp-python', llama_cpp.__version__, '| GPU offload:', llama_cpp.llama_supports_gpu_offload())"],
                   env=env, capture_output=True, text=True)
print(r.stdout + r.stderr[-1500:])


In [ ]:
from google.colab import userdata

API_KEY = "ganti-kunci-rahasia"   # kunci untuk endpoint publik
MODEL = "release"                  # "release" = MicroMind.gguf dari repo micromind, atau path .gguf di Drive

os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)
MODEL_PATH = "models/model.gguf"
if MODEL == "release":
    subprocess.run(["curl", "-L", "-s", "-o", MODEL_PATH,
                    "https://github.com/clovenowgaming-sudo/micromind/releases/latest/download/MicroMind.gguf"], check=True)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(MODEL, MODEL_PATH)
print("model: %.2f GB" % (os.path.getsize(MODEL_PATH) / 1e9))

H = {"Authorization": "Bearer " + userdata.get("GITHUB_TOKEN"), "Accept": "application/vnd.github+json"}
rel = requests.get("https://api.github.com/repos/clovenowgaming-sudo/geode-dataset/releases/latest", headers=H).json()
for a in rel["assets"]:
    r = requests.get(a["url"], headers={**H, "Accept": "application/octet-stream"})
    open("data/" + a["name"], "wb").write(r.content)
    print(a["name"], len(r.content))


In [ ]:
%%writefile rag_core.py
import os, re, json, gzip, math, hashlib
from collections import defaultdict

DATA = os.environ.get("DATA_DIR", "data")
CHUNK = 1400
TOPK = int(os.environ.get("TOPK", "5"))
OK_LIC = {"BSL-1.0", "MIT", "Apache-2.0", "Unlicense"}

SYS = ("Kamu adalah MicroMind, asisten AI untuk membuat mod Geode (Geometry Dash, Geode SDK 5.11.2, C++). "
       "Gunakan HANYA nama kelas, fungsi, dan makro yang muncul di REFERENSI di bawah atau yang benar-benar kamu yakini ada di Geode. "
       "Jangan mengarang API. Jika tidak ada di referensi dan kamu tidak yakin, katakan tidak yakin. "
       "Tulis kode lengkap yang bisa dikompilasi, jelaskan singkat, dan jawab dengan bahasa pengguna.")

TOK = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
CAMEL = re.compile(r"[A-Z]+(?![a-z])|[A-Z]?[a-z]+|\d+")


def toks(s):
    out = []
    for w in TOK.findall(s):
        out.append(w.lower())
        parts = [p.lower() for p in CAMEL.findall(w.replace("_", " "))]
        if len(parts) > 1:
            out.extend(parts)
    return [t for t in out if len(t) > 1]


def split_text(src, body):
    res, cur = [], ""
    for para in re.split(r"\n\s*\n", body):
        while len(para) > CHUNK * 2:
            if cur.strip():
                res.append((src, cur))
                cur = ""
            res.append((src, para[:CHUNK]))
            para = para[CHUNK:]
        if len(cur) + len(para) > CHUNK and cur.strip():
            res.append((src, cur))
            cur = ""
        cur += para + "\n\n"
    if cur.strip():
        res.append((src, cur))
    return res


class Index:
    def __init__(self):
        self.docs, self.boost, self.dl = [], [], []
        self.post = defaultdict(list)
        self.seen = set()
        self.N, self.avg = 0, 1.0

    def add(self, src, text, boost=1.0):
        h = hashlib.md5(text.encode("utf-8", "ignore")).hexdigest()
        if h in self.seen:
            return
        self.seen.add(h)
        t = toks(src + " " + text)
        if not t:
            return
        i = len(self.docs)
        self.docs.append((src, text))
        self.boost.append(boost)
        self.dl.append(len(t))
        tf = defaultdict(int)
        for w in t:
            tf[w] += 1
        for w, c in tf.items():
            self.post[w].append((i, c))

    def finish(self):
        self.N = len(self.docs)
        self.avg = sum(self.dl) / max(1, self.N)

    def search(self, q, k=TOPK):
        sc = defaultdict(float)
        for w in set(toks(q)):
            p = self.post.get(w)
            if not p:
                continue
            idf = math.log(1 + (self.N - len(p) + 0.5) / (len(p) + 0.5))
            for i, c in p:
                sc[i] += idf * c * 2.5 / (c + 1.5 * (0.25 + 0.75 * self.dl[i] / self.avg))
        ranked = sorted(sc.items(), key=lambda x: -x[1] * self.boost[x[0]])
        out, seen = [], set()
        for i, _ in ranked[:80]:
            key = re.sub(r"\s+", " ", self.docs[i][1])[:160]
            if key in seen:
                continue
            seen.add(key)
            out.append(self.docs[i])
            if len(out) >= k:
                break
        return out


def load_index():
    idx = Index()
    p = os.path.join(DATA, "geode-docs-5.11.2.md")
    if os.path.exists(p):
        text = open(p, encoding="utf-8").read()
        for part in text.split("<!-- FILE: ")[1:]:
            name, _, body = part.partition(" -->")
            for src, ch in split_text("docs/" + name.strip(), body.strip()):
                idx.add(src, ch, 1.4)
    p = os.path.join(DATA, "geode-api-headers-5.11.2.txt")
    if os.path.exists(p):
        text = open(p, encoding="utf-8").read()
        parts = re.split(r"\n\n// ===== FILE: (.+?) =====\n", text)
        for j in range(1, len(parts) - 1, 2):
            for src, ch in split_text("api/" + parts[j], parts[j + 1]):
                idx.add(src, ch, 1.2)
    p = os.path.join(DATA, "geode-code.jsonl.gz")
    if os.path.exists(p):
        with gzip.open(p, "rt", encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                repo, path = r["repo"], r["path"]
                if path.endswith(".bro"):
                    continue
                if repo == "geode-sdk/geode" and path.startswith(("loader/include", "bindings/")):
                    continue
                sdk = repo.startswith("geode-sdk/")
                if not (sdk or r["license"] in OK_LIC):
                    continue
                for src, ch in split_text("%s/%s" % (repo, path), r["text"]):
                    idx.add(src, ch, 1.1 if sdk else 1.0)
    idx.finish()
    return idx


def augment(messages, idx):
    q, extra = "", []
    for m in messages:
        if m.get("role") == "system" and isinstance(m.get("content"), str):
            extra.append(m["content"])
    for m in reversed(messages):
        if m.get("role") == "user":
            c = m.get("content")
            q = c if isinstance(c, str) else " ".join(x.get("text", "") for x in c if isinstance(x, dict))
            break
    hits = idx.search(q)
    ctx = "\n\n".join("[%d] %s\n%s" % (i + 1, s, t.strip()) for i, (s, t) in enumerate(hits))
    sys_text = SYS + ("\n\n" + "\n".join(extra) if extra else "") + "\n\nREFERENSI (potongan docs dan kode Geode SDK 5.11.2):\n" + ctx
    rest = [m for m in messages if m.get("role") != "system"]
    return [{"role": "system", "content": sys_text}] + rest


In [ ]:
%%writefile rag_server.py
import os
import httpx
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
import rag_core

LLAMA = os.environ.get("LLAMA_URL", "http://127.0.0.1:8081")
API_KEY = os.environ.get("API_KEY", "")

print("Membangun indeks RAG...", flush=True)
IDX = rag_core.load_index()
print("Indeks siap: %d potongan" % IDX.N, flush=True)

app = FastAPI()
client = httpx.AsyncClient(timeout=None)


def check(req: Request):
    if API_KEY and req.headers.get("authorization", "") != "Bearer " + API_KEY:
        raise HTTPException(status_code=401, detail="API key salah")


@app.get("/")
async def root():
    return {"status": "ok", "model": "micromind", "chunks": IDX.N}


@app.get("/v1/models")
async def models(request: Request):
    check(request)
    return {"object": "list", "data": [{"id": "micromind", "object": "model"}]}


@app.get("/search")
async def search(request: Request, q: str):
    check(request)
    return [{"source": s, "preview": t[:200]} for s, t in IDX.search(q)]


@app.post("/v1/chat/completions")
async def chat(request: Request):
    check(request)
    body = await request.json()
    body["messages"] = rag_core.augment(body.get("messages", []), IDX)
    if not body.get("max_tokens"):
        body["max_tokens"] = 1024
    if body.get("stream"):
        async def gen():
            async with client.stream("POST", LLAMA + "/v1/chat/completions", json=body) as r:
                async for chunk in r.aiter_raw():
                    yield chunk
        return StreamingResponse(gen(), media_type="text/event-stream")
    r = await client.post(LLAMA + "/v1/chat/completions", json=body)
    return JSONResponse(r.json(), status_code=r.status_code)


In [ ]:
subprocess.run("pkill -f llama_cpp.server; pkill -f uvicorn", shell=True)
os.environ["API_KEY"] = API_KEY
os.environ["DATA_DIR"] = "data"

llama = subprocess.Popen([V + "/bin/python", "-m", "llama_cpp.server", "--model", MODEL_PATH,
                          "--model_alias", "micromind", "--n_gpu_layers", "-1", "--n_ctx", "16384",
                          "--chat_format", "chatml", "--host", "127.0.0.1", "--port", "8081"],
                         stdout=open("llama.log", "w"), stderr=subprocess.STDOUT, env=env)

def tunggu(url, n):
    for _ in range(n):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False

if not tunggu("http://127.0.0.1:8081/v1/models", 120):
    print(open("llama.log").read()[-2500:])
    raise RuntimeError("server model gagal start")
print("server model siap")

rag = subprocess.Popen(["python", "-m", "uvicorn", "rag_server:app", "--host", "127.0.0.1", "--port", "8080"],
                       stdout=open("rag.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
if not tunggu("http://127.0.0.1:8080/", 240):
    print(open("rag.log").read()[-2500:])
    raise RuntimeError("server RAG gagal start")
print("server RAG siap")


In [ ]:
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
tun = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8080", "--no-autoupdate"],
                       stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        URL = m.group(0)
        break
    time.sleep(2)
print("URL publik:", URL)
print("base_url =", str(URL) + "/v1", "| API key =", API_KEY)


In [ ]:
r = requests.post("http://127.0.0.1:8080/v1/chat/completions",
                  headers={"Authorization": "Bearer " + API_KEY},
                  json={"messages": [{"role": "user", "content": "Buat hook MenuLayer::init di Geode yang menulis log."}],
                        "max_tokens": 400})
print(r.json()["choices"][0]["message"]["content"])


In [ ]:
while True:   # biarkan sel ini berjalan agar sesi tidak idle; tekan stop untuk mematikan
    time.sleep(300)
    print(time.strftime('%H:%M'), 'masih jalan', flush=True)
